# Scanpy Single-Cell RNA-seq Working Example

This notebook is a reproducible working example of a Scanpy single-cell RNA-seq workflow. It loads public 10x Multiome gene-expression count matrices, performs QC, doublet scoring, normalization, highly variable gene selection, PCA, nearest-neighbor graph construction, UMAP visualization, Leiden clustering, marker-gene analysis, and manual cell-type annotation.

The rendered artifact writes concrete outputs to `output/`: processed `.h5ad` files, QC metadata, marker-gene tables, cell-type annotation tables, summary UMAP figures, and package-version information. The notebook is intended as an executable example artifact rather than explanatory course material.


## Imports and Output Directories


In [ ]:
# Core scverse libraries
from __future__ import annotations

from pathlib import Path
import importlib.metadata as importlib_metadata
import sys

import anndata as ad
import matplotlib.pyplot as plt
import pandas as pd

# Data retrieval
import pooch
import scanpy as sc


In [ ]:
output_dir = Path("output")
figures_dir = output_dir / "figures"
output_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

sc.settings.figdir = figures_dir
sc.settings.set_figure_params(dpi=80, facecolor="white")


## Data Loading

The input data are public bone marrow mononuclear cell samples from healthy human donors, originally included in the Open Problems NeurIPS 2021 benchmarking dataset {cite}`Luecken2021`. The samples were measured with the 10x Multiome Gene Expression and Chromatin Accessibility kit.

The working example reads two 10x HDF5 count matrices into `AnnData`, makes feature and observation names unique, records the sample label in `.obs["sample"]`, and concatenates the samples into one analysis object.


In [ ]:
EXAMPLE_DATA = pooch.create(
    path=pooch.os_cache("scanpy_working_example"),
    base_url="doi:10.6084/m9.figshare.22716739.v1/",
)
EXAMPLE_DATA.load_registry_from_doi()


In [ ]:
samples = {
    "s1d1": "s1d1_filtered_feature_bc_matrix.h5",
    "s1d3": "s1d3_filtered_feature_bc_matrix.h5",
}
adatas = {}

for sample_id, filename in samples.items():
    path = EXAMPLE_DATA.fetch(filename)
    sample_adata = sc.read_10x_h5(path)
    sample_adata.var_names_make_unique()
    adatas[sample_id] = sample_adata

adata = ad.concat(adatas, label="sample")
adata.obs_names_make_unique()
print(adata.obs["sample"].value_counts())
adata

The combined object contains roughly 8,000 cells per sample and 36,601 measured genes. The following sections run a compact preprocessing, clustering, and annotation workflow suitable for demonstrating a reproducible Scanpy analysis artifact.


## Quality Control

`scanpy.pp.calculate_qc_metrics` computes common quality-control metrics based on detected genes, total counts, and user-defined gene groups. Mitochondrial, ribosomal, and hemoglobin genes are defined by prefix below so their count fractions can be tracked in `.obs`.


In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

Inspect the computed QC metrics:

* number of genes detected per cell
* total counts per cell
* percentage of counts in mitochondrial genes


In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

Inspect QC metrics jointly with a scatter plot colored by mitochondrial count fraction.


In [ ]:
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

The example applies a permissive initial filter: remove cells with fewer than 100 detected genes and remove genes detected in fewer than 3 cells. In a production analysis, these thresholds should be revisited per sample because QC distributions can differ substantially across batches.


In [ ]:
sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_genes(adata, min_cells=3)

### Doublet Detection

The workflow runs Scrublet through `scanpy.pp.scrublet`. Scrublet predicts cell doublets using a nearest-neighbor classifier over observed transcriptomes and simulated doublets, then stores `doublet_score` and `predicted_doublet` in `.obs`.


In [ ]:
sc.pp.scrublet(adata, batch_key="sample")

The working example keeps doublet annotations in the object rather than immediately filtering them. This preserves the information for later QC review alongside clusters and UMAP structure.


Alternative doublet-detection methods in the scverse ecosystem include [DoubletDetection](https://github.com/JonathanShor/DoubletDetection) and [SOLO](https://docs.scvi-tools.org/en/stable/user_guide/models/solo.html).


## Normalization

The workflow saves raw counts in `adata.layers["counts"]`, applies count-depth normalization with `scanpy.pp.normalize_total`, and then log-transforms the matrix with `scanpy.pp.log1p`.


In [ ]:
# Saving count data
adata.layers["counts"] = adata.X.copy()

In [ ]:
# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data
sc.pp.log1p(adata)

## Feature Selection

The workflow selects the 2,000 most informative genes with `scanpy.pp.highly_variable_genes`, stratified by sample. These genes are used for downstream dimensionality reduction and graph construction.


In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key="sample")

In [ ]:
sc.pl.highly_variable_genes(adata)

## Dimensionality Reduction

Principal component analysis (PCA) reduces the dimensionality of the expression matrix and provides a denoised representation for neighbor graph construction.


In [ ]:
sc.tl.pca(adata)

The variance-ratio plot documents the number of PCs considered for downstream neighbor graph construction and clustering.


In [ ]:
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

Inspect principal components for possible technical drivers such as sample, mitochondrial fraction, or other QC metrics.


In [ ]:
sc.pl.pca(
    adata,
    color=["sample", "sample", "pct_counts_mt", "pct_counts_mt"],
    dimensions=[(0, 1), (2, 3), (0, 1), (2, 3)],
    ncols=2,
    size=2,
)

## Nearest-Neighbor Graph Construction and UMAP Visualization

The workflow computes a nearest-neighbor graph from the PCA representation.


In [ ]:
sc.pp.neighbors(adata)

The graph is embedded in two dimensions with UMAP {cite:p}`McInnes2018`.


In [ ]:
sc.tl.umap(adata)

Visualize the UMAP by sample to inspect potential batch structure.


In [ ]:
sc.pl.umap(
    adata,
    color="sample",
    # Setting a smaller point size to get prevent overlap
    size=2,
)

The two samples show only a modest sample-associated structure in this example, so the workflow proceeds to clustering and annotation. For stronger batch effects, integration methods such as [`scanorama`](https://github.com/brianhie/scanorama) or [`scvi-tools`](https://scvi-tools.org) would be appropriate to evaluate.


## Clustering

As with Seurat and many other frameworks, we recommend the Leiden graph-clustering method (community detection based on optimizing modularity) {cite}`Traag2019`. Note that Leiden clustering directly clusters the neighborhood graph of cells, which we already computed in the previous section.

In [ ]:
# Using the igraph implementation and a fixed number of iterations can be significantly faster,
# especially for larger datasets
sc.tl.leiden(adata, flavor="igraph", n_iterations=2)

In [ ]:
sc.pl.umap(adata, color=["leiden"])

Saved plots are generated in the final artifact-export cell. Individual Scanpy plots can also be saved by using `show=False` followed by `matplotlib.pyplot.savefig(...)`.


## Reassess Quality Control and Cell Filtering

QC metrics are revisited on UMAP to check whether doublet scores, mitochondrial counts, total counts, or detected-gene counts align with specific clusters.


In [ ]:
sc.pl.umap(
    adata,
    color=["leiden", "predicted_doublet", "doublet_score"],
    # increase horizontal space between panels
    wspace=0.5,
    size=3,
)

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden", "log1p_total_counts", "pct_counts_mt", "log1p_n_genes_by_counts"],
    wspace=0.5,
    ncols=2,
)

## Manual cell-type annotation

This section performs compact manual cell-type annotation using marker genes. A production annotation pass would typically involve additional marker checks, subclustering, reference-based annotation, and expert review.


Cell-type annotation maps unsupervised clusters to biological labels using marker genes. This working example shows a lightweight marker-based pass sufficient for a reusable Scanpy artifact.


At this stage the object contains QC metrics, doublet scores, normalized expression, PCA, a neighbor graph, UMAP coordinates, and initial clusters. Marker genes are used below to connect clusters to broad cell identities.


Generate Leiden clusterings at several resolutions so marker expression can be compared across coarse and finer partitions.


In [ ]:
for res in [0.02, 0.5, 2.0]:
    sc.tl.leiden(adata, key_added=f"leiden_res_{res:4.2f}", resolution=res, flavor="igraph")

The number of clusters is controlled by the Leiden `resolution` parameter. The selected resolution should be justified by cluster stability, marker expression, and biological interpretability.


In [ ]:
sc.pl.umap(
    adata,
    color=["leiden_res_0.02", "leiden_res_0.50", "leiden_res_2.00"],
    legend_loc="on data",
)

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden_res_0.02", "leiden_res_0.50", "leiden_res_2.00"],
    legend_loc="on data",
)

UMAPs should not be overinterpreted, but they are useful for checking whether coarse, medium, and fine clustering resolutions produce biologically plausible groups.


### Marker gene set

Define marker genes for the main expected cell types. These markers were adapted from the [Single Cell Best Practices annotation chapter](https://www.sc-best-practices.org/cellular_structure/annotation.html).


In [ ]:
marker_genes = {
    "CD14+ Mono": ["FCN1", "CD14"],
    "CD16+ Mono": ["TCF7L2", "FCGR3A", "LYN"],
    # Note: DMXL2 should be negative
    "cDC2": ["CST3", "COTL1", "LYZ", "DMXL2", "CLEC10A", "FCER1A"],
    "Erythroblast": ["MKI67", "HBA1", "HBB"],
    # Note HBM and GYPA are negative markers
    "Proerythroblast": ["CDK6", "SYNGR1", "HBM", "GYPA"],
    "NK": ["GNLY", "NKG7", "CD247", "FCER1G", "TYROBP", "KLRG1", "FCGR3A"],
    "ILC": ["ID2", "PLCG2", "GNLY", "SYNE1"],
    "Naive CD20+ B": ["MS4A1", "IL4R", "IGHD", "FCRL1", "IGHM"],
    # Note IGHD and IGHM are negative markers
    "B cells": [
        "MS4A1",
        "ITGB1",
        "COL4A4",
        "PRDM1",
        "IRF4",
        "PAX5",
        "BCL11A",
        "BLK",
        "IGHD",
        "IGHM",
    ],
    "Plasma cells": ["MZB1", "HSP90B1", "FNDC3B", "PRDM1", "IGKC", "JCHAIN"],
    # Note PAX5 is a negative marker
    "Plasmablast": ["XBP1", "PRDM1", "PAX5"],
    "CD4+ T": ["CD4", "IL7R", "TRBC2"],
    "CD8+ T": ["CD8A", "CD8B", "GZMK", "GZMA", "CCL5", "GZMB", "GZMH", "GZMA"],
    "T naive": ["LEF1", "CCR7", "TCF7"],
    "pDC": ["GZMB", "IL3RA", "COBLL1", "TCF4"],
}

In [ ]:
sc.pl.dotplot(adata, marker_genes, groupby="leiden_res_0.02", standard_scale="var")

The coarsest clustering separates broad lineages that can be labeled from marker-gene expression.


In [ ]:
adata.obs["cell_type_lvl1"] = adata.obs["leiden_res_0.02"].map(
    {
        "0": "Lymphocytes",
        "1": "Monocytes",
        "2": "Erythroid",
        "3": "B Cells",
    }
)

In [ ]:
sc.pl.dotplot(adata, marker_genes, groupby="leiden_res_0.50", standard_scale="var")

Resolution 0.50 is used for cluster-level marker-gene analysis. A full analysis would inspect each cluster in more detail and subcluster where needed.


### Differentially-expressed Genes as Markers

Cluster-specific marker genes are computed with a Wilcoxon test for each cluster versus the rest of the cells.


In [ ]:
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(adata, groupby="leiden_res_0.50", method="wilcoxon")

Visualize the top 5 differentially expressed genes per cluster.


In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, groupby="leiden_res_0.50", standard_scale="var", n_genes=5)

Marker genes can then be linked to known cell biology. For example, a cluster expressing [*NKG7*](https://www.genecards.org/cgi-bin/carddisp.pl?gene=NKG7&keywords=nkg7) and [*GNLY*](https://www.genecards.org/cgi-bin/carddisp.pl?gene=GNLY&keywords=GNLY) is consistent with NK cells.


Differentially expressed genes can be extracted as a dataframe with `scanpy.get.rank_genes_groups_df`.


In [ ]:
sc.get.rank_genes_groups_df(adata, group="7").head(5)

In [ ]:
dc_cluster_genes = sc.get.rank_genes_groups_df(adata, group="7").head(5)["names"]
sc.pl.umap(
    adata,
    color=[*dc_cluster_genes, "leiden_res_0.50"],
    legend_loc="on data",
    frameon=False,
    ncols=3,
)

The p-values produced here can be extremely small because the test treats each cell as an independent sample. For a more conservative production analysis, consider pseudo-bulking by sample and cell type, then using a differential-expression tool such as [`pydeseq2`](https://pydeseq2.readthedocs.io/).


## Export Working-Example Artifacts


In [ ]:
# Export reproducible working-example artifacts.
marker_table = sc.get.rank_genes_groups_df(adata, group=None)
marker_table.to_csv(output_dir / "scanpy_cluster_markers_leiden_res_0.50.csv", index=False)

obs_columns = [
    col
    for col in [
        "sample",
        "n_genes_by_counts",
        "total_counts",
        "pct_counts_mt",
        "pct_counts_ribo",
        "pct_counts_hb",
        "doublet_score",
        "predicted_doublet",
        "leiden",
        "leiden_res_0.02",
        "leiden_res_0.50",
        "leiden_res_2.00",
        "cell_type_lvl1",
    ]
    if col in adata.obs
]
adata.obs[obs_columns].to_csv(output_dir / "scanpy_obs_qc_clusters_annotations.csv")

cell_type_table = (
    adata.obs[["leiden_res_0.02", "cell_type_lvl1"]]
    .drop_duplicates()
    .sort_values("leiden_res_0.02")
)
cell_type_table.to_csv(output_dir / "scanpy_cell_type_annotations.csv", index=False)

sc.pl.umap(
    adata,
    color=["sample", "leiden_res_0.50", "cell_type_lvl1"],
    ncols=3,
    show=False,
)
plt.savefig(figures_dir / "scanpy_umap_summary.png", bbox_inches="tight", dpi=150)
plt.close()

adata.write_h5ad(output_dir / "scanpy_bmmc_working_example.h5ad")

version_lines = [
    f"python: {sys.version.split()[0]}",
    f"scanpy: {sc.__version__}",
    f"anndata: {ad.__version__}",
    f"pandas: {pd.__version__}",
]
for package in ["numpy", "scipy", "matplotlib", "pooch"]:
    try:
        version_lines.append(f"{package}: {importlib_metadata.version(package)}")
    except importlib_metadata.PackageNotFoundError:
        version_lines.append(f"{package}: not installed")

(output_dir / "package_versions.txt").write_text("\n".join(version_lines) + "\n")
